# Find and Display Risk of Bias (RoB) Tables

This script demonstrates the enhanced `ExportReader` capabilities for identifying
Risk of Bias tables using "smart search" based on references and citations.

%% [markdown]
## Import Required Libraries

In [1]:
import pandas as pd
from pathlib import Path
from IPython.display import display, HTML

# Import the export reader library
from scraper.export_reader import load_latest_export

## Load the Export Data

In [2]:
# Load the latest export
try:
    reader = load_latest_export()
    reader.print_summary()
except Exception as e:
    print(f"Error loading export: {e}")
    # Fallback or exit if no export found
    exit(1)

Loading latest export: processed_export_1780342516.json
EXPORT SUMMARY
Total documents: 283
Export file: /home/jgrzyb/Documents/Python/ai4es-oa-paper-scrapper/notebooks/paper_pipeline_data/exports/processed_export_1780342516.json

Documents by source:
source
pmc      264
arxiv     19

Total tables: 2387
Total images: 1727

Document sections:
  Min sections: 0
  Max sections: 49
  Avg sections: 17.5



## Smart Search for RoB Tables
# 

Instead of searching for "method" or "risk of bias" in section titles, 
we search for sections that cite foundational RoB literature (Higgins 2011, Sterne 2019).

In [3]:
print("=" * 80)
print("SMART SEARCH: FINDING TABLES BY CITATION")
print("=" * 80 + "\n")

# Use the new find_rob_tables method
df_rob_tables = reader.find_rob_tables()

if df_rob_tables.empty:
    print("❌ No RoB tables found using the smart search.")
    print("This might be because the papers don't cite the standard RoB tools,")
    print("or the citations weren't extracted correctly.")
else:
    print(f"✅ Found {len(df_rob_tables)} tables citing Risk of Bias tools!\n")
    display(df_rob_tables[['paper_id', 'section', 'matched_rob_refs', 'csv_path']])

SMART SEARCH: FINDING TABLES BY CITATION

✅ Found 47 tables citing Risk of Bias tools!



,paper_id,section,matched_rob_refs,csv_path
0,PMC10258714,"Literature search, study characteristics and q...",[Paper cites RoB tools (heading match)],tables/PMC10258714/table_1.csv
1,PMC10516441,Methods,[B24],tables/PMC10516441/table_0.csv
2,PMC10516441,Methods,[B24],tables/PMC10516441/table_1.csv
3,PMC10516441,Methods,[B24],tables/PMC10516441/table_2.csv
4,PMC10646712,"Literature search, study characteristics, and ...",[Paper cites RoB tools (heading match)],tables/PMC10646712/table_1.csv
5,PMC11005196,Risk of bias results,[CR23],tables/PMC11005196/table_7.csv
6,PMC11187334,Risk of bias,[Paper cites RoB tools (heading match)],tables/PMC11187334/table_5.csv
7,PMC11193836,Study Selection,[CR34],tables/PMC11193836/table_4.csv
8,PMC11193836,Study Selection,[CR34],tables/PMC11193836/table_5.csv
9,PMC11193836,Study Selection,[CR34],tables/PMC11193836/table_6.csv


## Display Filtered Tables

In [4]:
def show_table(row_index, df_source):
    """Display a specific table from a given dataframe."""
    row = df_source.iloc[row_index]
    pid = row['paper_id']
    
    all_refs = reader.get_all_references_dataframe()
    ref_details = []
    
    for rid in row['matched_rob_refs']:
        if rid == "Paper cites RoB tools (heading match)":
            ref_details.append(f"<li><b>{rid}</b></li>")
            continue
            
        ref_info = all_refs[(all_refs['paper_id'] == pid) & (all_refs['ref_id'] == rid)]
        if not ref_info.empty:
            title = ref_info.iloc[0]['title']
            text = ref_info.iloc[0]['text']
            ref_details.append(f"<li><b>{rid}</b>: {title}<br><small>{text}</small></li>")
        else:
            ref_details.append(f"<li><b>{rid}</b></li>")
            
    ref_list_html = f"<ul>{''.join(ref_details)}</ul>" if ref_details else "None"
    
    # Display metadata
    header = f"""
    <div style="background-color: #e8f4f8; padding: 10px; border-radius: 5px; margin-bottom: 10px; border-left: 5px solid #2980b9;">
        <b>RoB Table #{row_index + 1}</b> | Paper: <b>{pid}</b> | Section: <i>{row['section']}</i>
        <br><b>Matched References:</b>
        {ref_list_html}
    </div>
    """
    display(HTML(header))
    
    # Load and display the table
    try:
        df = reader.load_table_csv(row['csv_path'])
        display(df)
    except FileNotFoundError as e:
        print(f"❌ Error loading table: {e}")
    except Exception as e:
        print(f"❌ Unexpected error: {e}")
    
    print("-" * 80)

# Display identified tables
if not df_rob_tables.empty:
    display_limit = min(10, len(df_rob_tables))
    print(f"Displaying top {display_limit} RoB tables...\n")
    
    for i in range(display_limit):
        try:
            show_table(i, df_rob_tables)
        except Exception as e:
            print(f"Error displaying table {i}: {e}")
            continue

Displaying top 10 RoB tables...



,No.,Author,Olopatadine group,Ketotifen group,Jada scores
0,Number,Age [years],Male (n),Methods,Number
1,1,Ul Abidin 2022,31,30.944 ±3.34,–
2,2,Sah 2019,30,24.16 ± 10.22,18
3,3,Patel 2018,60,36.35 ± 11.91,38
4,4,Mortemousque 2014,37,46.6 ± 18.5,13
5,5,Figus 2010,30,37 ±20,15
6,6,Borazan 2009,20,26.9 ± 10.6,10
7,7,Avunduk 2005,16,–,9


--------------------------------------------------------------------------------


,Inclusion criteria,Exclusion criteria
0,Patients older than 18 years,Patients younger than 18 years
1,AERD with or without comorbidities,No AERD
2,ESS (with or without AD + ATAD),Biologics
3,Papers 2016–2023,Older papers
4,"Prospective and retrospective studies, reviews...",Case study descriptions


--------------------------------------------------------------------------------


...

--------------------------------------------------------------------------------


,Study,Treatment arm,Comparator arm,Female sex,Age,Condition requiring penicillin
0,Iammatteo et al. (2019) [25],2-step direct DC to oral amoxicillin (n = 155),PST followed by a challenge to amoxicillin (n ...,DC: 77.4% (120)PST: 80% (136),DC: 50.1 ± 2.4PST: 52.1 ± 1.5,Not specified
1,Mustafa et al. (2019) [26],2-step direct DC to oral amoxicillin (n = 79),PST followed by challenge to amoxicillin (n = 80),69.8% (111),38.2 ± 25.0,Not specified
2,Stevenson et al. (2019) [27],1 or 2-step direct DC to oral amoxicillin or c...,PST followed by challenge to amoxicillin or cu...,DC: 55.7% (93)PST: 68.6% (192),DC: 42.4 ± 19.5PST: 47.0 ± 18.3,Not specified
3,Copaescu et al. (2023) (PALACE) [28],"1 or 2-step direct DC with oral amoxicillin, p...",PST followed by 1-step oral challenge (n = 190),65.5% (247),51 (35-66),Not specified
4,Ramsey et al. (2023) [29],2-step direct DC with oral amoxicillin (n = 16),PST followed by a challenge to amoxicillin (n ...,100% (38),28.4 (25.2–30.8),Peripartum prophylaxis in group B strept-colon...


--------------------------------------------------------------------------------


,Study,Drug-challenge,Skin test,Timing of outcome assessment
0,Iammatteo et al. (2019) [25],All graded challenges were single-blind (patie...,PST were performed when appropriate before cha...,DC: 120 minPST: not specified61.9% (n = 96) pa...
1,Mustafa et al. (2019) [26],Oral DC: 1/10 of the target dose of amoxicilli...,PST: skin prick test to volar forearm using th...,DC: 66.7 ± 4.8 minPST: 72.7 ± 5.3 min
2,Stevenson et al. (2019) [27],Oral amoxicillin challenge was performed for p...,PST was selected for each patient according to...,DC: 60 to 90 minPST: not specifiedPhone follow...
3,Copaescu et al. (2023) (PALACE) [28],Oral DC: lowest available therapeutic dose at ...,PST: skin prick test with benzylpenicilloyl po...,DC: 1.8 h (IQR 1.3–3.7)PST: 2.3 (IQR 1.7–5.5)A...
4,Ramsey et al. (2023) [29],N.A,N.A,DC 70 minPST 72 min


--------------------------------------------------------------------------------


## Reference Inspection
# 

Let's see what RoB references were found in the documents.

In [5]:
print("\n" + "=" * 80)
print("INSPECTING IDENTIFIED RoB REFERENCES")
print("=" * 80 + "\n")

df_refs = reader.get_all_references_dataframe()
if not df_refs.empty:
    # Filter for references that match our RoB criteria
    rob_dois = ['10.1136/bmj.d5928', '10.1136/bmj.l4898', 'd5928', 'l4898']
    rob_keywords = ['risk of bias', 'cochrane collaboration', 'rob 2', 'rob2']
    
    is_rob = df_refs['doi'].fillna('').str.lower().apply(
        lambda x: any(d.lower() in x for d in rob_dois)
    ) | df_refs['text'].fillna('').str.lower().apply(
        lambda x: any(k in x for k in rob_keywords)
    )
    
    found_rob_refs = df_refs[is_rob]
    print(f"Found {len(found_rob_refs)} unique RoB-related bibliographic entries across documents.")
    display(found_rob_refs[['paper_id', 'ref_id', 'doi', 'title', 'year']])
else:
    print("No references found in the export.")



INSPECTING IDENTIFIED RoB REFERENCES

Found 97 unique RoB-related bibliographic entries across documents.


,paper_id,ref_id,doi,title,year
299,PMC11694227,B24,10.1136/bmj.l4898,Rob 2: a revised tool for assessing risk of bi...,2019
638,PMC10420062,B7,10.1136/bmj.l4898,Rob 2: a revised tool for assessing risk of bi...,2019
836,PMC11370728,B37,NaN,NaN,2020
1035,PMC8588837,bib20,10.1016/j.jclinepi.2011.11.014,Assessing risk of bias in prevalence studies: ...,2012
1432,PMC9483654,bib18,NaN,NaN,2011
...,...,...,...,...,...
15464,PMC9236485,cit0023,10.1136/bmj.l4898,RoB 2: a revised tool for assessing risk of bi...,2019
15465,PMC9236485,cit0024,10.1136/bmj.i4919,ROBINS-I: a tool for assessing risk of bias in...,2016
15494,PMC9326918,cit0020,10.1136/bmj.d5928,The Cochrane Collaboration’s tool for assessin...,2011
16265,PMC11415968,bib57,10.1136/bmj.l4898,RoB 2: a revised tool for assessing risk of bi...,Aug 28 2019
